# Phase 6: Race-Free Sensitivity Analysis

This notebook compares the final XGBoost model with an otherwise identical model that excludes the NHANES race/ethnicity variable (`RIDRETH1`).

It reports:

- repeated five-fold cross-validation repeated five times;
- bootstrap 95% confidence intervals;
- testing-budget performance;
- bidirectional NHANES cycle holdout validation;
- direct changes in AUROC, AUPRC, Brier score, and case detection.

Upload `Cystatin_C_Phase3_Predictor_Outputs.zip` when prompted.

In [ ]:
%pip -q install xgboost

In [ ]:
import json, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

ROOT = Path("/content/cystatin_c_phase6")
INPUT = ROOT / "input"
OUTPUT = ROOT / "outputs"
INPUT.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

In [ ]:
from google.colab import files

uploaded = files.upload()
zip_names = [name for name in uploaded if name.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError("Upload exactly one Phase 3 predictor ZIP.")

zip_path = INPUT / zip_names[0]
zip_path.write_bytes(uploaded[zip_names[0]])

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(INPUT)

df = pd.read_csv(INPUT / "nhanes_final_predictor_dataset.csv")
print("Rows:", len(df))
print("Cases:", int(df["DISCORDANCE_30_CALIBRATED"].sum()))

In [ ]:
OUTCOME = "DISCORDANCE_30_CALIBRATED"
WEIGHT = "WTSCY4YR"

continuous = [
    "RIDAGEYR", "CREATININE_CALIBRATED", "LBXSBU", "LBXSAL",
    "LBXSGL", "BMI", "WAIST_CM", "MEAN_SBP", "MEAN_DBP",
    "HEMOGLOBIN", "CRP_MG_DL", "UACR_MG_G"
]

race_containing_categorical = [
    "RIAGENDR", "RIDRETH1", "DIAGNOSED_DIABETES",
    "DIAGNOSED_HYPERTENSION", "SMOKING_STATUS", "ANY_CVD"
]

race_free_categorical = [
    "RIAGENDR", "DIAGNOSED_DIABETES",
    "DIAGNOSED_HYPERTENSION", "SMOKING_STATUS", "ANY_CVD"
]

y = df[OUTCOME].astype(int)
weights = pd.to_numeric(df[WEIGHT], errors="coerce").fillna(1.0)

In [ ]:
def make_model(categorical):
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), continuous),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), categorical),
    ])

    return Pipeline([
        ("preprocess", preprocessor),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=3,
            learning_rate=0.03,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


def evaluate(y_true, probability):
    return {
        "AUROC": roc_auc_score(y_true, probability),
        "AUPRC": average_precision_score(y_true, probability),
        "Brier": brier_score_loss(y_true, probability),
    }


def bootstrap_ci(y_true, probability, n_bootstrap=2000):
    rng = np.random.default_rng(RANDOM_STATE)
    y_array = np.asarray(y_true)
    p_array = np.asarray(probability)
    values = {"AUROC": [], "AUPRC": [], "Brier": []}

    for _ in range(n_bootstrap):
        index = rng.integers(0, len(y_array), len(y_array))
        y_boot = y_array[index]
        p_boot = p_array[index]
        if len(np.unique(y_boot)) < 2:
            continue
        current = evaluate(y_boot, p_boot)
        for metric, value in current.items():
            values[metric].append(value)

    estimates = evaluate(y_array, p_array)
    rows = []
    for metric in values:
        rows.append({
            "Metric": metric,
            "Estimate": estimates[metric],
            "Lower 95% CI": np.percentile(values[metric], 2.5),
            "Upper 95% CI": np.percentile(values[metric], 97.5),
        })
    return pd.DataFrame(rows)


def testing_budget(y_true, probability, budgets=(0.10, 0.20, 0.30, 0.50)):
    y_array = np.asarray(y_true)
    p_array = np.asarray(probability)
    order = np.argsort(-p_array)
    total_cases = y_array.sum()
    rows = []

    for fraction in budgets:
        number_tested = int(np.ceil(len(y_array) * fraction))
        cases = y_array[order[:number_tested]].sum()
        rows.append({
            "Testing budget (%)": 100 * fraction,
            "People tested": number_tested,
            "Cases detected": int(cases),
            "Case detection (%)": 100 * cases / total_cases,
            "Positive yield (%)": 100 * cases / number_tested,
            "Number needed to test": number_tested / cases,
        })
    return pd.DataFrame(rows)

In [ ]:
specifications = {
    "Race-containing XGBoost": race_containing_categorical,
    "Race-free XGBoost": race_free_categorical,
}

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=RANDOM_STATE
)

predictions = {}
fold_rows = []

for label, categorical in specifications.items():
    features = continuous + categorical
    X = df[features]
    prediction_matrix = np.zeros((len(df), 5))

    for split_number, (train_index, test_index) in enumerate(cv.split(X, y)):
        repeat = split_number // 5
        fitted = make_model(categorical)
        fitted.fit(
            X.iloc[train_index],
            y.iloc[train_index],
            model__sample_weight=weights.iloc[train_index].to_numpy(),
        )
        probability = fitted.predict_proba(X.iloc[test_index])[:, 1]
        prediction_matrix[test_index, repeat] = probability

        row = evaluate(y.iloc[test_index], probability)
        row.update({
            "Model": label,
            "Repeat": repeat + 1,
            "Fold": split_number % 5 + 1,
        })
        fold_rows.append(row)

    predictions[label] = prediction_matrix.mean(axis=1)

fold_results = pd.DataFrame(fold_rows)

In [ ]:
ci_tables = []
budget_tables = []

for label, probability in predictions.items():
    current_ci = bootstrap_ci(y, probability)
    current_ci.insert(0, "Model", label)
    ci_tables.append(current_ci)

    current_budget = testing_budget(y, probability)
    current_budget.insert(0, "Model", label)
    budget_tables.append(current_budget)

bootstrap_results = pd.concat(ci_tables, ignore_index=True)
budget_results = pd.concat(budget_tables, ignore_index=True)

display(bootstrap_results.round(3))
display(budget_results.round(2))

In [ ]:
cycle_rows = []

for label, categorical in specifications.items():
    features = continuous + categorical
    X = df[features]

    for train_cycle, test_cycle in [
        ("1999-2000", "2001-2002"),
        ("2001-2002", "1999-2000"),
    ]:
        train = df["CYCLE"].eq(train_cycle)
        test = df["CYCLE"].eq(test_cycle)

        fitted = make_model(categorical)
        fitted.fit(
            X.loc[train],
            y.loc[train],
            model__sample_weight=weights.loc[train].to_numpy(),
        )
        probability = fitted.predict_proba(X.loc[test])[:, 1]
        row = evaluate(y.loc[test], probability)
        row.update({
            "Model": label,
            "Train cycle": train_cycle,
            "Test cycle": test_cycle,
            "Test N": int(test.sum()),
            "Test cases": int(y.loc[test].sum()),
        })
        cycle_rows.append(row)

cycle_results = pd.DataFrame(cycle_rows)
display(cycle_results.round(3))

In [ ]:
bootstrap_results.to_csv(
    OUTPUT / "race_free_bootstrap_performance.csv", index=False
)
budget_results.to_csv(
    OUTPUT / "race_free_testing_budgets.csv", index=False
)
cycle_results.to_csv(
    OUTPUT / "race_free_cycle_holdout.csv", index=False
)

prediction_output = pd.DataFrame({
    "SEQN": df["SEQN"],
    "Observed": y,
    **predictions,
})
prediction_output.to_csv(
    OUTPUT / "race_free_cross_validated_predictions.csv", index=False
)

output_zip = Path("/content/Cystatin_C_Phase6_Race_Free_Outputs.zip")
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT.iterdir():
        zf.write(file_path, arcname=file_path.name)

print("Created:", output_zip)
files.download(str(output_zip))